# **第48回日本高血圧学会総会** #
# **FP委員会企画05: アカデミアにおけるAI活用の原則と実践**

# **「機械学習を用いたテーブルデータ解析の基本と生成AIの学術への応用」**

日時: 2026年10月12日(月・祝) 8:20-9:30


【ご注意】
*   本ノートブックは上記ハンズオンのみを目的として作成されています。
*   本コードを実データや研究等に利用される場合は、利用者自身の責任において実施してください。
*   本コードの実行によって生じたいかなる不具合・損害についても、日本高血圧学会ならびに講演者は一切の責任を負いかねます。
*   本資料の転載・再配布はご遠慮ください。

# **[ハンズオン]機械学習を行うプログラムを触ってみよう**
# **①Google colaboratoryを使ったpythonnの演習環境の構築**

スライド58/94枚目: 数値の計算と表示

In [ ]:
１＋１

In [ ]:
1+1

スライド59/94枚目: 変数への数の代入と出力

In [ ]:
a = 1 + 4

In [ ]:
a = 2 + 7

スライド60/94枚目: 変数への数の代入と出力・四則演算

In [ ]:
b = 3 - 4
b
c = 15 * 3
c
d = 20 / 2
d

In [ ]:
b = 3 - 4
print(b)
c = 15 * 3
print(c)
d = 20 / 2
print(d) #最後の行なので d のみでも出力される

スライド61/94枚目: 割り算の商のみ・余剰のみの表示・べき乗

In [ ]:
print(15 // 2) #整数除算
print(7 % 3)   #剰余演算
print(2 ** 8)  #べき乗演算

# **②機械学習で構築する分類モデル：ロジスティック回帰分析**

スライド64/94枚目: ロジスティック回帰の実践:0.事前準備

In [ ]:
!pip install japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

スライド65−66/94枚目: ロジスティック回帰の実践:1.データの中身を確認する

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
bc

In [ ]:
print(bc.DESCR)

スライド68−69/94枚目: ロジスティック回帰の実践:3.特徴量と要約値の確認

In [ ]:
x_bc10 = bc.data.iloc[:,0:10]
x_bc10.head()

In [ ]:
x_bc10.describe()

In [ ]:
y_bc = bc.target
y_bc.head()

In [ ]:
y_bc.describe()

スライド70−72/94枚目: ロジスティック回帰の実践:4.単変量モデルを構築しテスト

In [ ]:
x_radius = x_bc10[['mean radius']]
print(x_radius.shape)
print(type(x_radius))

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_radius, y_bc, test_size=0.2, random_state=100)

In [ ]:
from sklearn.linear_model import LogisticRegression    # 前回はここが LinearRegression でした
model_bc_radius = LogisticRegression()

model_bc_radius.fit(x_train, y_train)

In [ ]:
print(model_bc_radius.coef_)
print(model_bc_radius.intercept_)

In [ ]:
print(model_bc_radius.predict(x_test))
print(np.array(y_test))

In [ ]:
print(model_bc_radius.predict_proba(x_test))

スライド73/94枚目: プロット(図示)のコマンド

In [ ]:
y_pred = model_bc_radius.predict(x_test)

# 決定境界の計算
coef = model_bc_radius.coef_[0][0]
intercept = model_bc_radius.intercept_[0]
decision_boundary_x = -intercept / coef

results_df = pd.DataFrame({
    'mean radius': x_test['mean radius'],
    'true_label': y_test,
    'predicted_label': y_pred
})

plt.figure(figsize=(10, 7))

# Y軸のラベルを反転させるため、プロットするy座標を調整する
# 元のラベル 0 (悪性) はプロット上は 1 に、元のラベル 1 (良性) はプロット上は 0 に配置

# 真陽性 (実際1: 良性, 予測1: 良性) -> プロットy座標: 0
tp = results_df[(results_df['true_label'] == 1) & (results_df['predicted_label'] == 1)]
plt.scatter(tp['mean radius'], np.zeros_like(tp['mean radius']), color='green', marker='o', label='真陽性 (実際1: 良性, 予測1: 良性)', alpha=0.7)

# 真陰性 (実際0: 悪性, 予測0: 悪性) -> プロットy座標: 1
tn = results_df[(results_df['true_label'] == 0) & (results_df['predicted_label'] == 0)]
plt.scatter(tn['mean radius'], np.ones_like(tn['mean radius']), color='purple', marker='o', label='真陰性 (実際0: 悪性, 予測0: 悪性)', alpha=0.7)

# 偽陽性 (実際0: 悪性, 予測1: 良性) - タイプIエラー -> プロットy座標: 1
fp = results_df[(results_df['true_label'] == 0) & (results_df['predicted_label'] == 1)]
plt.scatter(fp['mean radius'], np.ones_like(fp['mean radius']), color='orange', marker='x', s=100, label='偽陽性 (実際0: 悪性, 予測1: 良性)', alpha=0.9)

# 偽陰性 (実際1: 良性, 予測0: 悪性) - タイプIIエラー -> プロットy座標: 0
fn = results_df[(results_df['true_label'] == 1) & (results_df['predicted_label'] == 0)]
plt.scatter(fn['mean radius'], np.zeros_like(fn['mean radius']), color='red', marker='x', s=100, label='偽陰性 (実際1: 良性, 予測0: 悪性)', alpha=0.9)

# シグモイド曲線の計算
x_values_sigmoid = np.linspace(x_test['mean radius'].min() - 1, x_test['mean radius'].max() + 1, 500)
y_values_sigmoid = 1 / (1 + np.exp(-(coef * x_values_sigmoid + intercept)))
# Y軸反転に合わせてシグモイド曲線のプロットy座標も反転させる
plt.plot(x_values_sigmoid, 1 - y_values_sigmoid, color='blue', linestyle='-', label='シグモイド曲線 (予測確率)')

# 決定境界をプロット
plt.axvline(x=decision_boundary_x, color='black', linestyle='--', label=f'決定境界 (平均半径 = {decision_boundary_x:.2f})')

plt.title('ロジスティック回帰の性能: 真値と予測値')
plt.xlabel('平均半径')
plt.ylabel('ラベル (0: 悪性, 1: 良性)')
# Y軸の表示順序を反転
plt.yticks([0, 1], ['良性 (1)', '悪性 (0)'])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

スライド74/94枚目: 精度評価

In [ ]:
# 正解率のチェック
print(model_bc_radius.score(x_test, y_test))

# **③様々な機械学習を用いた分類モデルの作成:SVM、決定木**
# **サポートベクターマシン(SVM)**

  スライド83/94枚目: Python上でのSVMの実際

In [ ]:
# 解析に必要なモジュールの読み込み
import numpy as np
import pandas as pd

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc2 = bc.data.iloc[:,0:2]
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc2, y_bc, test_size=0.2, random_state=100)

# 学習モデルとしてサポートベクターマシン(SVM)を選択する
from sklearn.svm import SVC
model_svm = SVC(kernel='rbf', random_state=0)   # 今回は非線形の分離境界を設定したいのでkernelは'rbf'を指定する

# 学習データを入れて学習させる
model_svm.fit(x_train, y_train)

In [ ]:
# テスト用データで予測値と正解値の比較
print(model_svm.predict(x_test))
print(np.array(y_test))

In [ ]:
# モデルの精度を正解率(Accuracy)で評価
print(model_svm.score(x_test, y_test))

スライド85/94枚目: SVMを用いたモデルが描く非線形境界

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# グラフのサイズを設定
plt.figure(figsize=(10, 6))

# 各特徴量の最小値と最大値から、メッシュグリッド（描画領域）の範囲を決定
x_min, x_max = x_bc2.iloc[:, 0].min() - 1, x_bc2.iloc[:, 0].max() + 1
y_min, y_max = x_bc2.iloc[:, 1].min() - 1, x_bc2.iloc[:, 1].max() + 1

# 0.1刻みでメッシュグリッドを作成
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))

# メッシュグリッド上のすべての点に対して予測を行う
grid_data = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=x_bc2.columns)
Z = model_svm.predict(grid_data)
Z = Z.reshape(xx.shape)

# カラーマップの設定 (0:オレンジ色, 1:青色)
cmap_region = ListedColormap(['#ff7f0e', '#1f77b4'])

# 決定境界を等高線としてプロット
plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_region)

# テストデータの正解が0(malignant)のものをオレンジ色の三角印でプロット
plt.scatter(x_test.loc[y_test == 0].iloc[:, 0], x_test.loc[y_test == 0].iloc[:, 1],
            c='#ff7f0e', edgecolor='k', marker='^', label='Test Data (0: malignant)')

# テストデータの正解が1(benign)のものを青色の丸印でプロット
plt.scatter(x_test.loc[y_test == 1].iloc[:, 0], x_test.loc[y_test == 1].iloc[:, 1],
            c='#1f77b4', edgecolor='k', marker='o', label='Test Data (1: benign)')

plt.xlabel('mean radius')
plt.ylabel('mean texture')
plt.title('SVM Decision Boundary (Swapped Colors/Markers)')
plt.legend()
plt.show()

# **決定木(DT)**

スライド90/94枚目: Python上での決定木の実際

In [ ]:
# 解析に必要なモジュールの読み込み。今回はプロットもするのでmatplotlibも入れる。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc = bc.data     #今回は30特徴量すべてを使うので、.dataをそのままx_bcとする。
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc, y_bc, test_size=0.2, random_state=100)

# 学習モデルとして決定木(DT)を選択する。この際にハイパーパラメータも一緒に設定する。
from sklearn.tree import DecisionTreeClassifier
model_dt = DecisionTreeClassifier(
    criterion='gini',         # 純度評価法はジニ不純度('gini')を選択
    max_depth=2,              # ルートノードからリーフノードまでの深さの最大値は2に設定
    min_samples_split=10,      # 最低サンプルサイズは10に設定
    random_state=0)

# 学習データを入れて学習させる
model_dt.fit(x_train, y_train)

In [ ]:
# テスト用データで予測値と正解値の比較
print(model_dt.predict(x_test))
print(np.array(y_test))

In [ ]:
# モデルの精度を正解率(Accuracy)で評価
print(model_dt.score(x_test, y_test))

スライド92/94枚目: Python上での決定木の実際と図示

In [ ]:
from sklearn.tree import plot_tree
plot_tree(
    model_dt,
    feature_names=bc.feature_names,   # 特徴量の名前の一覧をNumPy配列で入力している
    class_names=bc.target_names,      # 正解値の分類名をNumPy配列で入力している
    filled=True,                      # Trueだとプロット図のノードに色をつける
    rounded=True,                     # ノードの角を丸めるかどうか
    fontsize=8,                       # プロット図のフォントの大きさを指定
)